# Handling Missing Values & Duplicates — Car Dataset

### 1. Load the car dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('car.xlsx')
print(df.shape)
df.head()

(999, 5)


,User ID,Gender,Age,AnnualSalary,Purchased
0,1,0.0,32.0,100000.0,1
1,2,1.0,38.0,63500.0,0
2,3,0.0,20.0,NaN,0
3,4,1.0,48.0,119000.0,1
4,5,0.0,41.0,108500.0,1


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   User ID       999 non-null    int64  
 1   Gender        983 non-null    float64
 2   Age           906 non-null    float64
 3   AnnualSalary  881 non-null    float64
 4   Purchased     999 non-null    int64  
dtypes: float64(3), int64(2)
memory usage: 39.2 KB


### 2. Number of null values in each field

In [3]:
df.isnull().sum()

User ID           0
Gender           16
Age              93
AnnualSalary    118
Purchased         0
dtype: int64

### 3. Null values only in AnnualSalary

In [4]:
df['AnnualSalary'].isnull().sum()

np.int64(118)

### 4. Drop rows having at least one null value in any field

In [5]:
df_any = df.dropna(how='any')
print('Before:', df.shape, '| After:', df_any.shape)

Before: (999, 5) | After: (775, 5)


### 5. Drop rows having null values in all fields

In [6]:
df_all = df.dropna(how='all')
print('Before:', df.shape, '| After:', df_all.shape)  # no row is completely empty

Before: (999, 5) | After: (999, 5)


### 6. Drop columns having at least one null value in any record

In [7]:
df_col_any = df.dropna(axis=1, how='any')
print('Remaining columns:', list(df_col_any.columns))
df_col_any.head()

Remaining columns: ['User ID', 'Purchased']


,User ID,Purchased
0,1,1
1,2,0
2,3,0
3,4,1
4,5,1


### 7. Drop columns having null values in all records

In [8]:
df_col_all = df.dropna(axis=1, how='all')
print('Remaining columns:', list(df_col_all.columns))  # no column is completely empty

Remaining columns: ['User ID', 'Gender', 'Age', 'AnnualSalary', 'Purchased']


### 8. Copy of dataset — replace nulls in AnnualSalary with 70000

In [9]:
df_copy = df.copy()
df_copy['AnnualSalary'] = df_copy['AnnualSalary'].fillna(70000)
print('Nulls in AnnualSalary (copy):', df_copy['AnnualSalary'].isnull().sum())
print('Nulls in AnnualSalary (original):', df['AnnualSalary'].isnull().sum())

Nulls in AnnualSalary (copy): 0
Nulls in AnnualSalary (original): 118


### 9. Replace nulls in Age with 0
*(done on a separate copy so the later median step still has real nulls to fill)*

In [10]:
df_age0 = df.copy()
df_age0['Age'] = df_age0['Age'].fillna(0)
print('Nulls in Age:', df_age0['Age'].isnull().sum())
print('Rows with Age = 0:', (df_age0['Age'] == 0).sum())

Nulls in Age: 0
Rows with Age = 0: 93


### 10. Replace nulls in AnnualSalary with the mean

In [11]:
salary_mean = df['AnnualSalary'].mean()
print('Mean AnnualSalary:', round(salary_mean, 2))
df['AnnualSalary'] = df['AnnualSalary'].fillna(salary_mean)
df['AnnualSalary'].isnull().sum()

Mean AnnualSalary: 74641.88


np.int64(0)

### 11. Replace nulls in Age with the median

In [12]:
age_median = df['Age'].median()
print('Median Age:', age_median)
df['Age'] = df['Age'].fillna(age_median)
df['Age'].isnull().sum()

Median Age: 40.0


np.int64(0)

### 12. Replace nulls in Gender with a suitable measure of central tendency
Gender is categorical (0/1), so mean and median don't make sense — the **mode** is the right measure.

In [13]:
gender_mode = df['Gender'].mode()[0]
print('Mode of Gender:', gender_mode)
df['Gender'] = df['Gender'].fillna(gender_mode)
df.isnull().sum()

Mode of Gender: 0.0


User ID         0
Gender          0
Age             0
AnnualSalary    0
Purchased       0
dtype: int64

### 13. Check for duplicates

In [14]:
print('Fully duplicated rows:', df.duplicated().sum())

# User ID is unique for every row, so also check ignoring it
print('Duplicates ignoring User ID:', df.duplicated(subset=['Gender','Age','AnnualSalary','Purchased']).sum())

Fully duplicated rows: 0
Duplicates ignoring User ID: 160


In [15]:
df[df.duplicated(subset=['Gender','Age','AnnualSalary','Purchased'], keep=False)].sort_values(['Age','AnnualSalary']).head(10)

,User ID,Gender,Age,AnnualSalary,Purchased
299,300,0.0,18.0,74641.884222,0
510,511,0.0,18.0,74641.884222,0
579,580,1.0,18.0,74641.884222,0
631,632,0.0,18.0,74641.884222,0
716,717,1.0,18.0,74641.884222,0
38,39,1.0,19.0,74641.884222,0
76,77,1.0,19.0,74641.884222,0
86,87,1.0,19.0,74641.884222,0
155,156,1.0,19.0,74641.884222,0
173,174,1.0,19.0,74641.884222,0


### 14. Remove the duplicate records

In [16]:
df_nodup = df.drop_duplicates()                        # exact duplicates
print('After drop_duplicates():', df_nodup.shape)

df_nodup = df.drop_duplicates(subset=['Gender','Age','AnnualSalary','Purchased'], keep='first')
print('After dropping duplicates ignoring User ID:', df_nodup.shape)
print('Remaining duplicates:', df_nodup.duplicated(subset=['Gender','Age','AnnualSalary','Purchased']).sum())

After drop_duplicates(): (999, 5)
After dropping duplicates ignoring User ID: (839, 5)
Remaining duplicates: 0
